In [ ]:
import h5py
import numpy as np
import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoConfig, AutoModelForMaskedLM, AutoTokenizer

# --- CONFIGURATION ---
MODEL_NAME = "RaphaelMourad/ModernBert-DNA-v1-37M-virus"
TASK_NAME = "dna_rna"
OUTPUT_H5 = f"{TASK_NAME}_ModernBERT-virus-37M.h5"

MAX_LEN = 4096
STRIDE = 512
BATCH_SIZE = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATASET ---
dataset = load_dataset("pabloarozarenad/vGUE-benchmark", TASK_NAME, split="test")
sequences = dataset["sequence"]
labels = [str(lbl) for lbl in dataset["label"]]
label_names = sorted(list(set(labels)))
label_to_id = {name: idx for idx, name in enumerate(label_names)}
label_ids = np.array([label_to_id[lbl] for lbl in labels], dtype=np.int64)

# --- 2. MODEL & TOKENIZER ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "left"

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.pad_token_id = tokenizer.pad_token_id

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.float32,
).to(DEVICE)
model.eval()

# --- 3. EXTRACTION ---
all_embeddings = []

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), BATCH_SIZE), desc="Extracting ModernBERT Embeddings"):
        batch_seqs = sequences[i : i + BATCH_SIZE]
        
        encoded = tokenizer(
            batch_seqs,
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            stride=STRIDE,
            return_overflowing_tokens=True,
            return_tensors="pt"
        )
        
        mapping = encoded["overflow_to_sample_mapping"].tolist()
        
        for sample_idx in range(len(batch_seqs)):
            window_indices = [win_idx for win_idx, src_idx in enumerate(mapping) if src_idx == sample_idx]
            window_embs = []
            
            for win_idx in window_indices:
                win_inputs = {
                    k: v[win_idx : win_idx + 1].to(DEVICE)
                    for k, v in encoded.items()
                    if k in ["input_ids", "attention_mask"]
                }
                
                # Extract Masked LM Head Logits
                logits = model(**win_inputs).logits  # Shape: [1, seq_len, 4096]
                win_emb = torch.max(logits, dim=1).values
                window_embs.append(win_emb.float().cpu())
            
            seq_emb = torch.mean(torch.cat(window_embs, dim=0), dim=0)
            all_embeddings.append(seq_emb.numpy())

final_matrix = np.vstack(all_embeddings)

# --- 4. SAVE ---
string_dtype = h5py.string_dtype(encoding="utf-8")
with h5py.File(OUTPUT_H5, "w") as f:
    f.create_dataset("embeddings", data=final_matrix)
    f.create_dataset("labels", data=np.array(labels, dtype=object), dtype=string_dtype)
    f.create_dataset("label_ids", data=label_ids)
    f.create_dataset("label_names", data=np.array(label_names, dtype=object), dtype=string_dtype)
    f.attrs["model_name"] = MODEL_NAME
    f.attrs["representation_type"] = "max_pooled_unmasked_mlm_head_logits"

print(f"Saved {final_matrix.shape} matrix to {OUTPUT_H5}")